# Notebook 01e: Redaction Impact Assessment
This notebook quantifies the extent of redaction in ASRS narratives. We search for placeholders like `ZZZ`, `XXX`, and `ZZZZ` to understand how much discriminative information might be lost.

In [ ]:
import pandas as pd
import yaml
import re
import matplotlib.pyplot as plt

with open('configs/main_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

df = pd.read_parquet(config['paths']['raw_data'])
print(f"Loaded {len(df)} records.")

## 1. Identify Redaction Placeholders

In [ ]:
redaction_pattern = re.compile(r'\b(Z{2,4}|X{2,4})\b', re.IGNORECASE)

def count_redactions(text):
    if pd.isna(text):
        return 0
    return len(redaction_pattern.findall(str(text)))

def get_token_count(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

df['redaction_count'] = df['Report 1_Narrative'].apply(count_redactions)
df['token_count'] = df['Report 1_Narrative'].apply(get_token_count)
df['redaction_pct'] = (df['redaction_count'] / df['token_count']).fillna(0) * 100

print(f"Average redaction percentage per report: {df['redaction_pct'].mean():.2f}%")

## 2. Visualize Redaction Impact

In [ ]:
plt.figure(figsize=(10, 6))
df['redaction_pct'].hist(bins=50)
plt.title('Distribution of Redaction Percentage in Narratives')
plt.xlabel('Redaction %')
plt.ylabel('Report Count')
plt.show()

## 3. Qualitative Review

In [ ]:
print("Sample of heavily redacted reports (>10% tokens redacted):")
heavy_redacted = df[df['redaction_pct'] > 10].sample(min(3, len(df[df['redaction_pct'] > 10])))

for i, row in heavy_redacted.iterrows():
    print(f"\nACN: {row.get('acn_num_ACN')} (Redaction: {row['redaction_pct']:.1f}%)")
    print(f"Narrative Snippet: {str(row.get('Report 1_Narrative'))[:500]}...")